# Dorso-ventral morphospace

Two publication panels from one integrated morphospace run, drawn combined and
as separate figures: A) shared dorsal/ventral
PCA colored and shaped by family, with representative images at the low and high
ends of both axes; B) rarefied dorsal and ventral disparity for the whole collection
and each family.

Species centroids require at least three photographs per side. PCA is fitted on
both sides together. Panel B uses the stored rarefied sum of variances in the full
embedding (mean and 95% interval over random subsets of the same number of species),
not the variance of the two plotted PCs. Colors are ColorBrewer Dark2. Database
access is read-only; no morphospaces are recomputed.

In [ ]:
import sys
from pathlib import Path

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "backend/app/configs/config.yaml").is_file()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
import matplotlib.pyplot as plt
from analyses.helpers.morphospace import (
    axis_ends,
    disparity_panel,
    morphospace_axes,
    morphospace_panel,
    morphospace_summaries,
    pca_axes,
)
from analyses.helpers.publication import (
    export_figure,
    load_settings,
    panel_left,
    publication_style,
)

settings = load_settings(ROOT)
publication_style()

# The scope panel A draws: the whole collection ("all", "all"), or one family,
# e.g. ("family", "nymphalidae").
SCOPE_RANK, SCOPE_KEY = "all", "all"

In [ ]:
summaries = morphospace_summaries(settings, scope_rank=SCOPE_RANK, scope_key=SCOPE_KEY)
scope = summaries["scope"].iloc[0]
print(
    f"Run {scope['run_id']}: {scope['scope_name']}, {scope['n_species']:,} species "
    f"({scope['n_species_both']:,} seen from both sides)"
)

## Combined morphospace figure

A) Species centroids on shared PC1 × PC2 axes, colored and shaped by family (by genus
when the scope is one family; groups past the six-color palette are gray "Other"):
dorsal filled, ventral hollow. Large outlined markers joined by a line mark each
family's mean dorsal and mean ventral position. The ends of each axis show the
same stored representative images as the site, set beside the y axis (high at the
top, low at the bottom) and below the x axis (low left, high right). Images are read
at full resolution from the backend's configured processed image directory; missing
examples raise an error.

B) Dot-and-whisker rarefied disparity: each dot is the mean sum of species-centroid
variances over random subsets of k species, and each whisker its 95% interval, for
the whole collection and every family, dorsal above ventral. Labels give the species
counts each side is resampled from. A side with fewer than k species is marked N/A,
not zero. The supporting CSV includes the unrarefied `sum_var` as well.

In [ ]:
fig = plt.figure(figsize=(20, 10), layout="constrained")
ax_a, x_strip, ax_b = morphospace_axes(fig)
morphospace_panel(ax_a, summaries)
disparity_panel(ax_b, summaries["disparity"])
# Titles sit above the legends; set before the layout so it makes room for them.
ax_a.set_title("A) Dorso-ventral morphospace", loc="left", fontsize=20, pad=62)
ax_b.set_title("B) Rarefied dorso-ventral disparity", loc="left", fontsize=20, pad=62)

# Resolve and freeze the layout, then place the axis-end images from where the
# axes ended up and move each title flush with its panel's leftmost drawing.
fig.canvas.draw()
fig.set_layout_engine(None)
strip_a = axis_ends(fig, ax_a, x_strip, summaries["extremes"], ROOT)
for ax, left in ((ax_a, strip_a.get_window_extent().x0), (ax_b, panel_left(ax_b))):
    box = ax.get_window_extent()
    ax.set_title(ax.get_title(loc="left"), loc="left", fontsize=20, pad=62, x=(left - box.x0) / box.width)

export_figure(fig, settings, "morphospace", summaries)
plt.show()
plt.close(fig)

## Separate figures

The same two panels as standalone figures, titled without panel letters:
`morphospace_pca` and `morphospace_disparity`, each exported with the summaries
it draws.

In [ ]:
fig = plt.figure(figsize=(11.5, 10), layout="constrained")
ax, x_strip = pca_axes(fig)
morphospace_panel(ax, summaries)
ax.set_title("Dorso-ventral morphospace", loc="left", fontsize=20, pad=62)

fig.canvas.draw()
fig.set_layout_engine(None)
strip = axis_ends(fig, ax, x_strip, summaries["extremes"], ROOT)
box = ax.get_window_extent()
ax.set_title(
    ax.get_title(loc="left"),
    loc="left",
    fontsize=20,
    pad=62,
    x=(strip.get_window_extent().x0 - box.x0) / box.width,
)

pca_summaries = {key: summaries[key] for key in ("scope", "points", "extremes")}
export_figure(fig, settings, "morphospace_pca", pca_summaries)
plt.show()
plt.close(fig)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9), layout="constrained")
disparity_panel(ax, summaries["disparity"])
ax.set_title("Rarefied dorso-ventral disparity", loc="left", fontsize=20, pad=36)

fig.canvas.draw()
fig.set_layout_engine(None)
box = ax.get_window_extent()
ax.set_title(
    ax.get_title(loc="left"),
    loc="left",
    fontsize=20,
    pad=36,
    x=(panel_left(ax) - box.x0) / box.width,
)

disparity_summaries = {key: summaries[key] for key in ("scope", "disparity")}
export_figure(fig, settings, "morphospace_disparity", disparity_summaries)
plt.show()
plt.close(fig)